In [72]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

_img_size = (48, 48)

def carregar_base_de_dados(caminho_principal, lista_classes, proporcao_teste):
    """
    Varre as subpastas no caminho_principal, processa as imagens e 
    retorna matrizes prontas para o treinamento.
    """
    imagens_processadas = []
    labels_numéricos = []
    
    mapeamento_classes = {nome_classe: idx for idx, nome_classe in enumerate(lista_classes)}
    
    print("🔄 Iniciando o carregamento e processamento das imagens...")
    
    for nome_classe in lista_classes:
        caminho_pasta = os.path.join(caminho_principal, nome_classe)
        
        if not os.path.isdir(caminho_pasta):
            print(f"⚠️ Aviso: A pasta '{nome_classe}' não foi encontrada. Pulando...")
            continue
            
        codigo_classe = mapeamento_classes[nome_classe]
        
        for nome_arquivo in os.listdir(caminho_pasta):
            caminho_imagem = os.path.join(caminho_pasta, nome_arquivo)
            img = cv2.imread(caminho_imagem)
            if img is None:
                continue
                
            # Pipeline de tratamento de imagem
            cinza = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            if cinza.shape != _img_size:
                cinza = cv2.resize(cinza, _img_size)
                
            _, binaria = cv2.threshold(cinza, 127, 255, cv2.THRESH_BINARY)
            # Normaliza e Centraliza os dados entre -0.5 e 0.5
            X_normalizado = (binaria / 255.0) - 0.5 
            
            # Achata a imagem de (48, 48) para um vetor de (2304,)
            imagens_processadas.append(X_normalizado.flatten())
            labels_numéricos.append(codigo_classe)
            
    X_dados = np.array(imagens_processadas)
    Y_dados = np.array(labels_numéricos)
    
    print(f"✅ Total carregado: {len(X_dados)} imagens.")
    
    X_train, X_test, Y_train, Y_test = train_test_split(
        X_dados, Y_dados, 
        test_size=proporcao_teste, 
        random_state=1, 
        stratify=Y_dados
    )
    
    print(f"📊 Divisão Concluída: Treino = {len(X_train)} | Teste = {len(X_test)}")
    return X_train, X_test, Y_train, Y_test

In [73]:
classes = ["Semibreve", "Minima", "Seminima", "Colcheia", "Semicolcheia"]
diretorio_base = r'C:\Users\jesse\SIN5016\dados_treino'

# Carrega e divide os dados
X_treino, X_teste, Y_treino, Y_teste = carregar_base_de_dados(diretorio_base, classes, proporcao_teste=0.2)

🔄 Iniciando o carregamento e processamento das imagens...
✅ Total carregado: 4250 imagens.
📊 Divisão Concluída: Treino = 3400 | Teste = 850


### Treino

In [74]:
def softmax(Z):
    # Subtrai o max por linha para estabilidade numérica (evita estouro do exp)
    exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

def treino(alpha, epocas, X, Y):

    num_amostras, num_features = X_treino.shape
    num_classes = len(classes)
    Y_one_hot = np.eye(num_classes)[Y]

    W = np.zeros((num_features, num_classes))
    B = np.zeros(num_classes)

    for epoca in range(epocas):
        Z = np.dot(X, W) + B
        Y_hat = softmax(Z)

        # Cálculo dos Gradientes
        dW = np.dot(X.T, (Y_hat - Y_one_hot)) / num_amostras
        db = np.sum(Y_hat - Y_one_hot, axis=0) / num_amostras

        W = W - (alpha * dW)
        B = B - (alpha * db)

        # perda = -np.mean(np.sum(Y_one_hot * np.log(np.clip(Y_hat, 1e-15, 1.0)), axis=1))
        # print(f"Época {epoca + 1}/{epocas} - Perda: {perda:.4f}")

        if (epoca + 1) % 10 == 0 or epoca == 0:
            # Perda por Entropia Cruzada Multiclasse
            perda = -np.mean(np.sum(Y_one_hot * np.log(np.clip(Y_hat, 1e-15, 1.0)), axis=1))
            print(f"Época {epoca + 1}/{epocas} - Perda: {perda:.4f}")

    return W, B
        

In [75]:
# Definiçao dos hiparâmetros
alpha = 0.005
epocas = 500

W, B = treino(alpha, epocas, X_treino, Y_treino)

Época 1/500 - Perda: 1.6094


Época 10/500 - Perda: 1.5418
Época 20/500 - Perda: 1.4784
Época 30/500 - Perda: 1.4244
Época 40/500 - Perda: 1.3784
Época 50/500 - Perda: 1.3389
Época 60/500 - Perda: 1.3044
Época 70/500 - Perda: 1.2741
Época 80/500 - Perda: 1.2472
Época 90/500 - Perda: 1.2229
Época 100/500 - Perda: 1.2010
Época 110/500 - Perda: 1.1809
Época 120/500 - Perda: 1.1624
Época 130/500 - Perda: 1.1453
Época 140/500 - Perda: 1.1294
Época 150/500 - Perda: 1.1146
Época 160/500 - Perda: 1.1007
Época 170/500 - Perda: 1.0876
Época 180/500 - Perda: 1.0752
Época 190/500 - Perda: 1.0635
Época 200/500 - Perda: 1.0524
Época 210/500 - Perda: 1.0418
Época 220/500 - Perda: 1.0317
Época 230/500 - Perda: 1.0221
Época 240/500 - Perda: 1.0128
Época 250/500 - Perda: 1.0040
Época 260/500 - Perda: 0.9955
Época 270/500 - Perda: 0.9873
Época 280/500 - Perda: 0.9795
Época 290/500 - Perda: 0.9719
Época 300/500 - Perda: 0.9646
Época 310/500 - Perda: 0.9575
Época 320/500 - Perda: 0.9507
Época 330/500 - Perda: 0.9441
Época 340/500 - Per

### Teste

In [76]:
def predict_proba(X):
    Z = np.dot(X, W) + B
    return softmax(Z)

def predict(X):
    probabilidades = predict_proba(X)
    return np.argmax(probabilidades, axis=1)

In [77]:
Y_pred_teste = predict(X_teste)
acuracia = np.mean(Y_pred_teste == Y_teste)
print(f"\n🎯 Acurácia no conjunto de teste: {acuracia * 100:.2f}%")


🎯 Acurácia no conjunto de teste: 71.18%


### Validação

In [78]:
diretorio_validacao = r'C:\Users\jesse\SIN5016\dados_validacao'

# Carrega e divide os dados (80% treino, 20% teste)
X_valid_treino, X_valid_teste, Y_valid_treino, Y_valid_teste = carregar_base_de_dados(diretorio_validacao, classes, proporcao_teste=0.2)

X_valid = X_valid_treino
Y_valid = Y_valid_treino

🔄 Iniciando o carregamento e processamento das imagens...
✅ Total carregado: 750 imagens.
📊 Divisão Concluída: Treino = 600 | Teste = 150


In [79]:
Y_pred_valid = predict(X_valid)
acuracia_valid = np.mean(Y_pred_valid == Y_valid)
print(f"\n🎯 Acurácia no conjunto de Validação: {acuracia_valid * 100:.2f}%")


🎯 Acurácia no conjunto de Validação: 64.83%


### Predição de uma nova Imagem

In [80]:
# === PREDIÇÃO EM UMA NOVA IMAGEM ===
caminho_nova_img = r'C:\Users\jesse\SIN5016\bases\seminima_3.jpg'

img = cv2.imread(caminho_nova_img)
cinza = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
if cinza.shape != _img_size:
    cinza = cv2.resize(cinza, _img_size)

_, binaria = cv2.threshold(cinza, 127, 255, cv2.THRESH_BINARY)
X_novo = ((binaria / 255.0) - 0.5).flatten().reshape(1, -1)

probabilidades = predict_proba(X_novo)[0]
classe_predita_idx = predict(X_novo)[0]
nome_classe_predita = classes[classe_predita_idx]

print("\n--- Resultado da Predição Individual ---")
for idx, classe_nome in enumerate(classes):
    print(f"Probabilidade de ser {classe_nome}: {probabilidades[idx]:.2%}")
print(f"\n🏆 Classificação Final: {nome_classe_predita}")


--- Resultado da Predição Individual ---
Probabilidade de ser Semibreve: 3.11%
Probabilidade de ser Minima: 20.53%
Probabilidade de ser Seminima: 45.92%
Probabilidade de ser Colcheia: 20.16%
Probabilidade de ser Semicolcheia: 10.29%

🏆 Classificação Final: Seminima


In [81]:
from sklearn.linear_model import LogisticRegression

# Treinando com a biblioteca otimizada (solver lbfgs ou saga)
clf = LogisticRegression(multi_class='multinomial', max_iter=500, C=0.005)
clf.fit(X_treino, Y_treino)
print("Acurácia Scikit-Learn:", clf.score(X_teste, Y_teste))

c:\Users\jesse\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Acurácia Scikit-Learn: 0.8035294117647059
